## import thư viện

In [ ]:
!pip install transformers==4.57.3 accelerate==1.12.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 686.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
%%writefile train.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import json

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments, EarlyStoppingCallback
from datasets import load_dataset, Dataset, concatenate_datasets

## import dataset ViNLI + large (250k)
df = pd.read_csv("/kaggle/input/datasets/huynhdat2010/rerank-finetune-data/ViNLI_and_large.csv")
df = Dataset.from_pandas(df)

# Shuffle dữ liệu
df = df.shuffle(seed=42)

# Kiểm tra thông tin
print(f"Size: {df.shape}")

## chia train/dev/test
split = df.train_test_split(test_size=0.003, seed=42)
temp_train = split["train"]
test_dataset = split["test"]

train_valid = temp_train.train_test_split(test_size=0.003, seed=42)
train_dataset = train_valid["train"]
valid_dataset = train_valid["test"]

print("Train:", len(train_dataset))
print("Valid:", len(valid_dataset))
print("Test:", len(test_dataset))

## tokenizer load
tokenizer = AutoTokenizer.from_pretrained(
    "Alibaba-NLP/gte-multilingual-reranker-base",
    trust_remote_code=True
)

def tokenize_fn(examples):
    res = tokenizer(
        examples["query"],
        examples["document"],
        truncation=True,
        # padding="max_length",
        max_length=512
    )
    res["labels"] = [float(x) for x in examples["label"]]
    return res

tokenized_train = train_dataset.map(tokenize_fn, batched=True, remove_columns=train_dataset.column_names)
tokenized_valid = valid_dataset.map(tokenize_fn, batched=True, remove_columns=valid_dataset.column_names)
tokenized_test  = test_dataset.map(tokenize_fn, batched=True, remove_columns=test_dataset.column_names)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8
)

## model
model = AutoModelForSequenceClassification.from_pretrained(
    "Alibaba-NLP/gte-multilingual-reranker-base",
    trust_remote_code=True
)

def compute_metrics(pred):
    labels = pred.label_ids
    scores = pred.predictions.squeeze()
    preds = (scores > 0).astype(int)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
        "f1": f1_score(labels, preds),
    }

training_args = TrainingArguments(
    output_dir="/kaggle/working/reranker_ckpt",
    overwrite_output_dir=True,

    eval_strategy="steps",
    save_strategy="steps",
    logging_strategy="steps",

    eval_steps=1000,
    save_steps=1000,
    logging_steps=200,

    learning_rate=2e-5,

    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    gradient_accumulation_steps=1,

    num_train_epochs=10,
    weight_decay=0.01,
    fp16=True,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    warmup_ratio=0.1,
    save_total_limit=1,
    load_best_model_at_end=True,
    report_to="none",
    label_names=["labels"],
    gradient_checkpointing=True,
    remove_unused_columns=False
)

### train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("------ Start Training ------")
trainer.train(
    resume_from_checkpoint="/kaggle/input/datasets/huynhdat2010/rerank-output-checkpoint/reranker_ckpt/checkpoint-6000"
)

### lưu
model.save_pretrained("/kaggle/working/rerank_model_finetuned_ViNLI_large")
tokenizer.save_pretrained("/kaggle/working/rerank_model_finetuned_ViNLI_large")
print("Model saved!")

Writing train.py


In [ ]:
!torchrun --nproc_per_node=2 train.py

W0525 04:11:50.530000 58 torch/distributed/run.py:792] 
W0525 04:11:50.530000 58 torch/distributed/run.py:792] *****************************************
W0525 04:11:50.530000 58 torch/distributed/run.py:792] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0525 04:11:50.530000 58 torch/distributed/run.py:792] *****************************************
2026-05-25 04:12:02.610699: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-25 04:12:02.610694: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779682322.772311      60 cuda_dnn.cc:8310] Unable to